# 10万条商品月度销售数据清洗与自动化报表生成

**作者**：[叶广]
**技术栈**：Python (Pandas, NumPy)
**业务背景**：模拟市场调研/咨询公司日常数据处理需求。针对包含重复记录和缺失值的 10 万条原始商品月度销售数据，进行清洗与自动化排版，解决人工 Excel 拉透视表耗时且易出错的问题。

## 核心处理流程
1. **防御性数据测试**：人为制造 500 条重复数据与 51 条空值，测试代码的鲁棒性。
2. **数据清洗**：统一数据类型，基于业务主键（商品ID + 日期）去重，并使用 `fillna(0)` 兜底补齐缺失的销售数量。
3. **类型转换**：将数量列转换为可空整数 `Int64`，保证数据严谨且避免小数显示。
4. **时间周期处理**：将字符串日期转为标准时间格式，并提取 `Period`（月度周期）。
5. **二维交叉表排版**：使用 `pivot_table` 按商品（行）和月份（列）聚合数据，输出标准的二维报表底表。

In [11]:
import pandas as pd
import numpy as np

# ==========================================
# 1. 模拟数据源
# ==========================================
df = pd.DataFrame({
    'product_id': np.random.choice(['P1001', 'P1002', 'P1003'], 100000),
    'sale_date': pd.date_range('2023-01-01', periods=100000, freq='h').astype(str),
    'quantity': np.random.choice([10.0, 20.0, 30.0, np.nan], 100000)
})

# 模拟真实业务中的脏数据
df = pd.concat([df, df.sample(500)], ignore_index=True)
df.loc[0:50, 'quantity'] = np.nan

# ==========================================
# 2. 数据清洗
# ==========================================
df['quantity'] = df['quantity'].astype('float')
df['sale_date'] = df['sale_date'].astype(object)

df = df.drop_duplicates(subset=['product_id', 'sale_date'], keep='first')
df['quantity'] = df['quantity'].fillna(0)
df['quantity'] = df['quantity'].astype('Int64')

df['sale_date'] = pd.to_datetime(df['sale_date'])
df['month_period'] = df['sale_date'].dt.to_period('M')

# ==========================================
# 3. 建交叉表（横排版）
# ==========================================
pivot_df = pd.pivot_table(
    df, values='quantity', index='product_id', columns='month_period', aggfunc='sum', fill_value=0
)
pivot_df.columns = [str(col) for col in pivot_df.columns]
pivot_df.index.name = '商品ID'
pivot_df.columns.name = '销售月份'

# 导出横排 Excel
pivot_df.to_excel('商品月度销售报表_横排.xlsx', index_label='商品ID/月份')
print("横排 Excel 已生成！")

# ==========================================
# 4. 使用for循环进行数据融化long
# ==========================================
# 创建空列表
result_list = []

# 遍历横版交叉表
for index, row in pivot_df.iterrows():
    product_id = index  # 拿到商品ID
    for month, quantity in row.items(): # 遍历这一行的所有月份和销量
        # 把每一条记录整理成字典，加到列表里
        result_list.append({
            '商品ID': product_id,
            '销售月份': month,
            '销售数量': quantity
        })

# 把列表转回 DataFrame，就变成竖排了！
df_long = pd.DataFrame(result_list)

# 排序（用老师教的 sort_values）
df_long = df_long.sort_values(by=['商品ID', '销售月份']).reset_index(drop=True)

print("\n--- 竖排版长表前 10 行 ---")
print(df_long.head(10))

# 导出竖排 Excel
df_long.to_excel('商品月度销售报表_竖排.xlsx', index=False)
print("竖排 Excel 已生成！")

横排 Excel 已生成！

--- 竖排版长表前 10 行 ---
    商品ID     销售月份  销售数量
0  P1001  2023-01  3280
1  P1001  2023-02  3140
2  P1001  2023-03  3570
3  P1001  2023-04  4000
4  P1001  2023-05  3810
5  P1001  2023-06  3230
6  P1001  2023-07  3390
7  P1001  2023-08  3750
8  P1001  2023-09  3770
9  P1001  2023-10  3490
竖排 Excel 已生成！
